<b>¡Hola Kevin!</b>

Mi nombre es Alejandro Abia y tengo el gusto de revisar tu proyecto.

A continuación, encontrarás mis comentarios en celdas pintadas de tres colores (verde, amarillo y rojo), a manera de semáforo. Por favor, <b>no las borres ni muevas de posición</b> mientras dure el proceso de revisión.

<div class="alert alert-block alert-success">
<b>Éxito</b> <a class="tocSkip"></a>
En celdas verdes encontrarás comentarios en relación a tus aciertos y fortalezas.
</div>
<div class="alert alert-block alert-warning">
<b>Atención</b> <a class="tocSkip"></a>
Utilizaré el color amarillo para llamar tu atención, expresar algo importante o compartirte alguna idea de valor.
</div>
<div class="alert alert-block alert-danger">
<b>A resolver</b> <a class="tocSkip"></a>
En rojo emitiré aquellos puntos que podrían impedir que el proyecto se ejecute correctamente. No son errores, sino oportunidades importantes de mejora.
</div>
<div class="alert alert-block alert-info">
<b>Comentario estudiante</b><a class="tocSkip"></a>
Si durante la revisión deseas dejarme algún comentario, por favor utiliza celdas azules como esta.
</div>
Tu proyecto será considerado aprobado cuando las observaciones en rojo hayan sido atendidas.
¡Empecemos!


## Descripción
La cadena de gimnasios Model Fitness está desarrollando una estrategia de interacción con clientes basada en datos analíticos.

Uno de los problemas más comunes que enfrentan los gimnasios y otros servicios es la pérdida de clientes. ¿Cómo descubres si un/a cliente ya no está contigo? Puedes calcular la pérdida en función de las personas que se deshacen de sus cuentas o no renuevan sus contratos. Sin embargo, a veces no es obvio que un/a cliente se haya ido: puede que se vaya de puntillas.

Los indicadores de pérdida varían de un campo a otro. Si un usuario o una usuaria compra en una tienda en línea con poca frecuencia, pero con regularidad, no se puede decir que ha huido. Pero si durante dos semanas no ha abierto un canal que se actualiza a diario, es motivo de preocupación: es posible que tu seguidor o seguidor/a se haya aburrido y te haya abandonado.

En el caso de un gimnasio, tiene sentido decir que un/a cliente se ha ido si no viene durante un mes. Por supuesto, es posible que estén en Cancún y retomen sus visitas cuando regresen, pero ese no es un caso típico. Por lo general, si un/a cliente se une, viene varias veces y luego desaparece, es poco probable que regrese.

Con el fin de combatir la cancelación, Model Fitness ha digitalizado varios de sus perfiles de clientes. Tu tarea consiste en analizarlos y elaborar una estrategia de retención de clientes.

## Objetivos
- Aprender a predecir la probabilidad de pérdida (para el próximo mes) para cada cliente.
- Elaborar retratos de usuarios típicos: selecciona los grupos más destacados y describe sus características principales.
- Analizar los factores que más impactan la pérdida.
- Sacar conclusiones básicas y elaborar recomendaciones para mejorar la atención al cliente:
- identificar a los grupos objetivo;
- sugerir medidas para reducir la rotación;
- describir cualquier otro patrón que observes con respecto a la interacción con los clientes.

In [1]:
#Importamos las librerías necesarias
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from scipy.cluster.hierarchy import linkage, dendrogram
from sklearn.cluster import KMeans

In [2]:
#Cargamos el CSV y lo convertimos a un DataFrame
gym = pd.read_csv('dataset/gym_churn_us.csv')

FileNotFoundError: [Errno 2] No such file or directory: 'dataset/gym_churn_us.csv'

## Paso 1 - EDA
### 1.1 - Llevar a cabo el análisis exploratorio de datos

In [3]:
#Realizamos un primer vistazo general al DataFrame
gym.info()

#Mostramos 6 filas de muestra del DataFrame
gym.sample(6)

NameError: name 'gym' is not defined

<div class="alert alert-block alert-success">
<b>Celda [3]</b> <a class="tocSkip"></a><br>
Excelente trabajo al realizar un primer vistazo al DataFrame utilizando `info()` y `sample()`. Esto te permite entender rápidamente la estructura de tus datos y detectar posibles problemas iniciales como tipos de datos incorrectos o valores atípicos.
</div>


In [4]:
#Buscamos valores duplicados en el DataFrame
print('Cantidad de valores duplicados:',gym.duplicated().sum())

#Buscamos valores únicos dentro de cada columna
unique_counts = gym.nunique()
print('Cantidad de valores únicos por columna:')
print(unique_counts)

#Convertimos a minúsculas los nombres de las columnas
gym.columns = gym.columns.str.lower()
gym.columns

NameError: name 'gym' is not defined

<div class="alert alert-block alert-success">
<b>Celda [4]</b> <a class="tocSkip"></a><br>
Has buscado valores duplicados y únicos.
</div>


    - En un primer vistazo la mayoria de las series cuentan con datos binarios, siendo solo las series que comienzan por AVG las que por ser de origen financiero tienen una alta varidedad entre sus valores.
    - No se identificaron datos duplicados en todo el dataframe y todas las series son del tipo adecuado para su posterior manejo.
    - La única modificación realizada fue la estandarización en los nombres de las series que pasaron a ser todas en minusculas.

### 1.2 - Observa el dataset: ¿contiene alguna característica ausente? Estudia los valores promedio y la desviación estándar (utiliza el método describe()).

In [5]:
#Buscamos valores nulos en el DataFrame
print('Cantidad de valores nulos:',gym.isnull().sum().sum())


#Aplicamos el método describe() para obtener estadísticas descriptivas del DataFrame
gym.describe()

NameError: name 'gym' is not defined

<div class="alert alert-block alert-success">
<b>Celda [5]</b> <a class="tocSkip"></a><br>
Muy bien al utilizar `describe()` para obtener estadísticas descriptivas. Esto proporciona una visión general de las distribuciones de tus datos y puede ayudarte a identificar rápidamente valores atípicos o distribuciones sesgadas.
</div>


    - No se encontraron valores nulos en ninguna de las series.
    - Al identificar que la mayoria de las series tienen valores binarios se vuelve facilmente identificable si el promedio se encuentra cercano al 0.5 entonces está balanceada la serie como en GENDER, PARTNER y GROUP_VISITS.
    - Tambien salta a la vista que las series mientras sean menores del balance binario la desviación se amplia como en CHURN, y ocurre el efecto contrario cuando el promedio esta muy por encima del balance como en NEAR_LOCARION.

### 1.3 - Observa los valores medios de las características en dos grupos: para las personas que se fueron (cancelación) y para las que se quedaron (utiliza el método groupby()).

In [6]:
#Dividimos el DataFrame en dos partes: una con el valor 1 en la columna 'churn' y otra sin ella con groupby
gym_churn = gym.groupby('churn').get_group(1)
gym_active = gym.groupby('churn').get_group(0)

#Analizamos nuevamente los nuevos DataFrames con el método describe()
display('Tabla con clientes inactivos (cancelación) del gimnasio:', gym_churn.describe())
display('Tabla con clientes activos del gimnasio:', gym_active.describe())

NameError: name 'gym' is not defined

<div class="alert alert-block alert-success">
<b>Celda [6]</b> <a class="tocSkip"></a><br>
Has segmentado correctamente los datos en clientes activos e inactivos. Sin embargo, sería útil visualizar la distribución de algunas variables clave en estos segmentos para identificar patrones distintos. Considera usar gráficos de cajas o violín para variables continuas.
</div>


    - Ahora que hemos dividido la serie entre los clientes Activos y los Inactivos (quienes cancelaron), encontramos cantidades similares en los promedios de la serie de GENDER, siendo la única con esta similitud.
    - Mientras que las que las series con mayor diferencia son: PARTNER, GROUP_VISITS, LIFETIME, CONTRACT_PERIOD, MONTH_TO_END_CONTRACT y LIFETIME, Estas series nos dan pistas sobre el porque los clientes cancelan su contrato al gimnasio.


### 1.4 - Traza histogramas de barras y distribuciones de características para aquellas personas que se fueron (cancelación) y para las que se quedaron.

In [7]:
#Creamos la función para trazar las gráficas de barras facilmente
#Recibe los parametros: column (columna a analizar), extra (texto extra para el eje x) y title (título de la gráfica)
def plot_bar( column, title_x, title):
    plt.figure(figsize=(8, 4))
    sns.countplot(data=gym, x=column, hue='churn', palette={0: 'blue', 1: 'red'})
    plt.title(title)
    plt.xlabel(title_x)
    plt.ylabel('Número de Clientes')
    plt.legend(title='Churn', loc='upper right', labels=['Activo', 'Inactivo'])
    plt.show()

#Creamos la función para trazar histogramas basados en los DF divididos por CHURN
#Recibe los parametros: column (columna a analizar), title_x (título del eje x) y title (título de la gráfica)
def plot_hist(column, title_x, title):
    plt.figure(figsize=(8, 4))
    sns.histplot(gym_churn[column], kde=True, color='red', label='Inactivos')
    sns.histplot(gym_active[column], kde=True, color='blue', label='Activos')
    plt.title(title)
    plt.xlabel(title_x)
    plt.ylabel('Frecuencia')
    plt.legend()
    plt.show()

In [8]:

#Trazamos una gráfica de barras para visualizar la distribución por genero
plot_bar('gender', 'Género','Distribución de usuarios por Género')


#Trazamos una gráfica de barras para visualizar la distribución por cercanía de trabajo/hogar al gimnasio
plot_bar('near_location', 'Distancia de hogar/trabajo al gym. 0=Lejos, 1=Cerca', 'Distribución de usuario por Cercanía de trabajo/hogar al Gimnasio')

#Trazamos una gráfica de barras para visualizar la distribución de usuarios con contrato de partner
plot_bar('partner', 'Tipo de contrato. 0=Sin Partner, 1=Con Partner', 'Distribución de usuarios con Contrato de Partner')


NameError: name 'gym' is not defined

<Figure size 800x400 with 0 Axes>

<div class="alert alert-block alert-success">
<b>Celda [8]</b> <a class="tocSkip"></a><br>
Has implementado de manera efectiva las funciones para graficar distribuciones categóricas. Esto es crucial para entender cómo se distribuyen las características categóricas en relación con el churn y puede ofrecer insights valiosos para el análisis.
</div>


    - Ahora que se graficaron las caracteristicas se aprecia con mayor facilidad las comportamientos similares y diferentes entre ambos dataframes. Confirmamos que el GENRE no muestra mucha diferencia entre ambos dataframes.
    - Los usuarios que viven/laboran cerca del gimnasio muestran un grado de cancelación mucho menor que los que se encuentran lejos.
    - Del total de los usuarios que NO tienen un contrato con partner el comportamiento de cancelación es praccticamente de la mitad, mientras que es practicamente una cuarta parte los que cancelan obteniendo el beneficio de este contrato.

In [9]:
#Trazamos una gráfica de barras para visualizar la distribución de usuarios que fueron referidos por un amigo
plot_bar('promo_friends', 'Referido por Amigos. 0=No, 1=Sí', 'Distribución de usuarios referidos por Amigos')

#Trazamos una gráfica de barras para visualizar la distribución por periodo de membresía
plot_bar('contract_period', 'Periodo de Membresía (en meses)', 'Distribución de usuarios por Periodo de Membresía')

#Trazamos una gráfica de barras para visualizar la distribución de usuarios que participan en sesiones grupales
plot_bar('group_visits', 'Participación en Sesiones Grupales. 0=No, 1=Sí', 'Distribución de usuarios que participan en Sesiones Grupales')

NameError: name 'gym' is not defined

<Figure size 800x400 with 0 Axes>

    - De los usuarios que recibieron el beneficio promocional de ser referidos por un amigo es solo una cuarta parte el proporcional que cancela, al contrario de los que NO fueron referidos que el proporcional es cercano a la mitad.
    - En la grafica de barras sobre el periodo de membresia encontramos que el porcentaje de cancelaciones disminuye en gran medida mientras más largo sea su contrato.
    - El proporcional de usuarios que cancelan el contrato pero que participan en una clase grupal dentro del gimnasio es menos de la mitad que los que NO.

In [10]:
#Trazamos histogramas para visualizar la distribución de edad de cada dataframe
plot_hist('age', 'Edad', 'Distribución de Edad de Clientes')

#Trazamos un histograma para visualizar la distribución del promedio de cargos extras de cada dataframe
plot_hist('avg_additional_charges_total', 'Promedio por Cargos Extras', 'Distribución del Promedio de Cargos Extras para Clientes Inactivos y Activos')

#Trazamos un histograma para visualizar la distribución de meses restantes del contrato de cada dataframe
plot_hist('month_to_end_contract', 'Meses restantes del Contrato', 'Distribución de Meses Restantes del Contrato para Clientes Inactivos y Activos')

NameError: name 'gym_churn' is not defined

<Figure size 800x400 with 0 Axes>

<div class="alert alert-block alert-success">
<b>Celda [10]</b> <a class="tocSkip"></a><br>
Buen trabajo, los histogramas son útiles para visualizar distribuciones de datos.
</div>


    - Resulta curioso que la distribución de los usuarios que cancelan son más jovenes que los que continuan activos, se observa una diferencia de alrededor de 5 años entre los picos de cada campana.
    - En la gráfica de la distribución de los cargos extras resulta relevante identificar que los usuarios Activos gastan mucho más del doble que los que cancelan.
    . En la tercera gráfica notamos que la mayor parte de los usuarios se concentran en 1 solo mes restante, sabemos que no todos esos usuarios tienen contrato de renovación mensual pero conforme a gráficas anteriores confirmamos que efectivamente gran parte de los usuarios manejan este tipo de renovación de contrato. Esto mismo hace que gran parte de los usuarios que cancelen pertenezcan a esta temporalidad.

In [11]:
#Trazamos un histograma sobre la distribución de usuarios por el tiempo (en meses) desde que el usuario visitó por primera vez
plot_hist('lifetime', 'Meses desde la primera visita', 'Distribución de usuarios desde su primera visita al gimnasio')

#Trazamos un histograma sobre la distribución de usuarios por la cantidad de visitas al gimnasio a la semana
plot_hist('avg_class_frequency_total', 'Visitas por Semana', 'Distribución de usuarios por Visitas al Gimnasio por Semana')

#Trazamos un histograma sobre la distribución de usuarios por el promedio de visitas por semana al gimnasio durante el mes en curso
plot_hist('avg_class_frequency_current_month', 'Visitas promedio a la semana durante el mes en curso', 'Distribución de usuarios por promedio de visitas a la semana al Gimnasio durante el mes en curso')

NameError: name 'gym_churn' is not defined

<Figure size 800x400 with 0 Axes>

    - En la gráfica de distribución de meses desde la primera visita de los usuarios identificamos facilmente que los usuarios normalmente cancelan durante su primer o segundo mes en el gimnasio y conforme avanza el tiempo los usuarios que cancelan disminuyen expnensialmente.
    - El comportamiento de la asistencia semanal de los usuarios es un gran indicador del porque los clientes cancelan, siendo que la mayoria de los inactivos asisten menos de 2 días a la semana mientras que los activos logran empujar esta distribución al asistir por lo menos 1 día más.

### 1.5 - Crea una matriz de correlación y muéstrala.

In [12]:
#Creamos una matriz de correlación para visualizar la relación entre las variables
plt.figure(figsize=(12, 8))
correlation_matrix = gym.corr()
display(correlation_matrix)

NameError: name 'gym' is not defined

<Figure size 1200x800 with 0 Axes>

<div class="alert alert-block alert-success">
<b>Celda [12]</b> <a class="tocSkip"></a><br>
La matriz de correlación es una herramienta poderosa para identificar relaciones entre variables. Has hecho un buen trabajo al visualizarla, lo que te permitirá detectar correlaciones significativas que podrían influir en el modelo predictivo.
</div>


    - En esta matriz de correlación se deben buscar las series cuyo coeficiente sea cercano a +1 o -1 para identificar las correlaciones.
    - Enfocados en la serie de CHURN y buscando identificar las razones por las cuales los clientes cancelan encontramos que solo las siguientes series cuentan con una correlación negativa mayor a -0.38 son CONTRAC_PERIOD, AGE, MONTH_TO_END_CONTRACT, AVG_CLASS_FREQUENCY_CURRENT_MONTH y la más alta es LIFETIME.
    - Cabe mencionar que la unica serie con la que CHURN tiene una correlación positiva es GENDER, sin embargo es tan cercana a 0 que no es tan relevante.

## Paso 2 - Construir un modelo para predecir la cancelación de usuarios
- Crea un modelo de clasificación binaria para clientes donde la característica objetivo es la marcha del usuario o la usuaria el mes siguiente.
- Recuerda indicar el parámetro random_state cuando dividas los datos y definas el algoritmo. 

### 2.1 - Divide los datos en conjuntos de entrenamiento y validación utilizando la función train_test_split().

In [13]:
#Dividimos los datos de entrenamiento y prueba
X = gym.drop(columns=['churn'])
y = gym['churn']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)

NameError: name 'gym' is not defined

<div class="alert alert-block alert-warning">
<b>Celda [13]</b> <a class="tocSkip"></a><br>
Al dividir los datos en conjuntos de entrenamiento y prueba, asegúrate de que el conjunto de prueba sea representativo del conjunto de datos completo. Puedes verificar esto comparando las estadísticas descriptivas de ambos conjuntos.
</div>


### 2.3 - Entrena el modelo en el set de entrenamiento con dos métodos:
#### 2.3.1 - Regresión logística

In [14]:
#Agregamos a las lista de modelos la regresión logística
models=[]
models = [LogisticRegression(random_state=0, max_iter=400)]

#### 2.3.2 - Bosque aleatorio.

In [15]:
#Agregamos el modelo de random forest
models.append(RandomForestClassifier(random_state=0, n_estimators=300))

### 2.4 - Evalúa la exactitud, precisión y recall para ambos modelos utilizando los datos de validación. Utilízalos para comparar los modelos. 
- ¿Qué modelo dio mejores resultados?

In [16]:
#Definimos la función para entrenar y evaluar los modelos
def train_and_evaluate_model(model, X_train, y_train, X_test, y_test):
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    roc_auc = roc_auc_score(y_test, model.predict_proba(X_test)[:, 1])
    print('Exactitud (Accuracy):{:.2f} Precisión:{:.2f} Sensabilidad(Recall):{:.2f} F1:{:.2f} AUC-ROC:{:.2f} '.format(
        accuracy_score(y_test, y_pred), precision_score(y_test, y_pred), recall_score(y_test, y_pred), f1_score(y_test, y_pred), roc_auc_score(y_test, y_pred)))

#Creamos un bucle para entrenar y evaluar cada modelo
for model in models:
    print(f'Modelo: {model.__class__.__name__}')
    train_and_evaluate_model(model, X_train, y_train, X_test, y_test)
    print('\n')

Modelo: LogisticRegression


NameError: name 'X_train' is not defined

<div class="alert alert-block alert-success">
<b>Celda [16]</b> <a class="tocSkip"></a><br>
Has implementado una función clara y concisa para entrenar y evaluar los modelos. Esto es una excelente práctica para mantener tu código organizado y reutilizable.
</div>


    - Una vez que comparamos ambos modelos encontramos una alta similitud entre sus métricas, siendo la REGRESIÓN LOGISTICA la que tiene una pequeña ventaja.
    - Cabe recordar que la REGRESIÓN  es un modelo más simple y rápido en comparación con el BOSQUE ALEATORIO, pero el modelo de BOSQUE se puede ajustar con mayor cantidad de parametros para refinar su proceso y quizas ofrecer mejores resultados.

## Paso 3. Crear clústeres de usuarios
Deja de lado la columna con datos sobre la cancelación e identifica los clústeres de objetos (usuarios/as):

### 3.1 - Estandariza los datos.

In [17]:
#Estandarizamos los datos
scaler = StandardScaler()
X_train_st = scaler.fit_transform(X_train)
X_test_st = scaler.transform(X_test)

NameError: name 'X_train' is not defined

<div class="alert alert-block alert-warning">
<b>Celda [17]</b> <a class="tocSkip"></a><br>
El escalado de características es crucial para algoritmos sensibles a las magnitudes de las variables, como la regresión logística. Sin embargo, asegúrate de que todas las características relevantes, especialmente las numéricas, estén correctamente escaladas antes de modelar.
</div>


### 3.2 - Utiliza la función linkage() para crear una matriz de distancias basada en la matriz de características estandarizada y trazar un dendrograma. Nota: ¡renderizar el dendrograma puede llevar tiempo! Utiliza el gráfico resultante para estimar el número de clústeres que puedes destacar.

In [18]:

#Creamos una matriz de distancias basada en la matriz estandarizada con la funcion linkage
Z = linkage(X_train_st, method='ward')

#Trazamos el dendrograma para visualizar la jerarquía de clusters
plt.figure(figsize=(18, 14))
dendrogram(Z, orientation='top')
plt.title('Dendrograma de Clustering Jerárquico')
plt.xlabel('Clientes')
plt.ylabel('Distancia')
plt.show()

NameError: name 'X_train_st' is not defined

### 3.3 - Entrena el modelo de clustering con el algortimo K-means y predice los clústeres de clientes. 
- Deja que el número de clústeres sea n=5 para que sea más sencillo comparar los resultados con los del resto del estudiantado. Sin embargo, en la vida real, nadie te dará tales pistas, así que tendrás que decidir basándote en el gráfico del paso anterior.)

In [19]:
#Entrenamos el modelos con k-means, definimos el modelo con 5 clústeres aunque si nos basaramos en el dendrograma, se identificaron 4 niveles (colores)
km = KMeans(n_clusters=5)
labels = km.fit_predict(gym)

#Agregamos las etiquetas de los clústeres al DataFrame de entrenamiento y agrupamos por el cluster
gym['cluster_km'] = labels

NameError: name 'gym' is not defined

<div class="alert alert-block alert-warning">
<b>Celda [19]</b> <a class="tocSkip"></a><br>
Al aplicar K-Means, es importante estandarizar las características para que las distancias sean comparables. Asegúrate de que el DataFrame utilizado para K-Means esté correctamente escalado. Además, podrías experimentar con diferentes números de clústeres y evaluar cuál ofrece la mejor segmentación.
</div>


### 3.4 - Mira los valores medios de característica para los clústeres.¿Hay algo que te llame la atención?

In [20]:
#Calculamos los valores medios de características para cada clúster
cluster_means = gym.groupby(['cluster_km']).mean()
display(cluster_means)

NameError: name 'gym' is not defined

    - Después de aplicar el algoritmo de K-Means en 5 clusters encontramos los siguientes caracterisiticas claves:
        * AVG_ADDITIONAL_CHARGES_TOTAL: es la caracteristica con la mayor diferencia entre los clusters.
        * CONTRACT_PERIOD: Los contratos más largos están en el cluster 3 que es el de mayor gasto.
        * PROMO_FRIENDS: El cluster 3 es el que tiene la más baja proporción de clientes por promoción.
        * AVG_CLASS_FREQUENCY_CURRENT_MONTH: El cluster 3 es el que cuenta con la mayor cantidad de usuarios Activos actualmente, al contrario el cluster 0 es el de menor cantidad.
    
    - Podemos identificar los clusters con mayor cantidad de usuarios Activos y sus caracteristicas, así como su contrario:
        * Cluster 3: Clientes de alta actividad, contratos largos, alto gasto adicional y casi sin promociones. Pueden ser clientes más fidelizados o premium.
        * Cluster 0: Clientes con contratos cortos, poco gasto adicional, y baja frecuencia actual. Pueden estar en riesgo de abandono.

### 3.5 - Traza distribuciones de características para los clústeres. ¿Notas algo?

In [21]:
#creamos una función para trazar los histogramas de los clústeres
def plot_cluster_histogram(column, title_x, title):
    plt.figure(figsize=(10, 6))
    sns.histplot(data=gym, x=column, hue='cluster_km', kde=True, palette='Set1', multiple='stack')
    plt.title(title)
    plt.xlabel(title_x)
    plt.ylabel('Cantidad de Clientes')
    plt.show()

#Creamos una lista con los nombres de las caracteristicas mas relevantes a analizar
features = ['contract_period', 'promo_friends', 'promo_friends', 'avg_class_frequency_total', 'month_to_end_contract']
#Iteramos sobre la lista de características y trazamos los histogramas
for feature in features:
    plot_cluster_histogram(feature, feature, f'Distribución de {feature} por Clústeres')

NameError: name 'gym' is not defined

<Figure size 1000x600 with 0 Axes>

### 3.6 - Calcula la tasa de cancelación para cada clúster (utiliza el método groupby()).
- ¿Difieren en términos de tasa de cancelación?
- ¿Qué grupos son propensos a irse y cuáles son leales?

In [22]:
#Calculamos la tasa de cancelación para cada clúster y los ordenamos de mayor a menor
churn_rate = gym.groupby('cluster_km')['churn'].mean()
print('Tasa de cancelación por clúster:')
print((churn_rate.sort_values(ascending=False).round(2))*100)

NameError: name 'gym' is not defined

    - Al calcular la tasa de cancelación por clusters ratificamos la explicación del contraste de desempeño entre clusters, siendo el cluster 0 el de la mayor cantidad de cancelaciones y el cluster 3 en el que debido a sus caracteristicas los clientes cancelan menos.

## Paso 4. Saca conclusiones y haz recomendaciones básicas sobre el trabajo con clientes
Llega a conclusiones y formula recomendaciones con respecto a la estrategia para la interacción y retención de clientes.

No necesitas entrar en detalles. Bastarán tres o cuatro principios esenciales y ejemplos de su implementación en forma de pasos de marketing específicos.

En este proyecto, desarrollamos modelos predictivos para estimar la probabilidad de pérdida (churn) de clientes para el próximo mes, logrando métricas de precisión sólidas con modelos como LogisticRegression y RandomForestClassifier. Los modelos permiten anticipar qué clientes tienen mayor riesgo de cancelar su membresía, lo cual es clave para implementar estrategias proactivas de retención. Además, mediante el uso de análisis de clustering, identificamos grupos diferenciados de clientes con comportamientos y necesidades específicas.

El análisis de segmentación nos permitió elaborar retratos de clientes típicos. Por ejemplo, detectamos un grupo altamente comprometido con contratos largos, frecuencia alta de visitas y gasto adicional elevado; estos clientes rara vez provienen de promociones. Por otro lado, identificamos un grupo en riesgo, con contratos cortos, baja frecuencia de clases recientes y poco gasto adicional. Los factores que más impactan la pérdida incluyen la duración del contrato, la frecuencia de visitas recientes, el gasto adicional y la participación en promociones con amigos. Clientes con baja actividad actual y contratos próximos a vencer presentan el mayor riesgo de abandono.

Como recomendaciones estratégicas, sugerimos focalizar los esfuerzos de marketing en los grupos en riesgo con campañas personalizadas como:
- Ofrecer extensiones de contrato con beneficios a clientes con contratos cortos y baja frecuencia reciente.
- Implementar programas de referencia y promociones para amigos en los segmentos más sensibles al precio.
- Diseñar comunicaciones personalizadas y seguimiento intensivo para clientes cuya frecuencia de asistencia haya disminuido en el último mes.
- Incentivar la participación en actividades grupales o clases especiales, especialmente en clientes con contratos cortos.

Estas acciones contribuirán a aumentar la fidelización y reducir la rotación de forma sostenible, permitiendo optimizar la estrategia de interacción con clientes y mejorar la experiencia global.

<div class="alert alert-block alert-success">
<b>Comentario final</b> <a class="tocSkip"></a><br>
    
¡Muy buen trabajo, Kevin! A lo largo del proyecto mostraste fortalezas muy claras:<br><br>

1. Realizaste una carga de datos eficiente y un análisis inicial exhaustivo.
2. Implementaste una limpieza de datos cuidadosa, incluyendo la detección de duplicados y valores nulos.
3. Segmentaste adecuadamente los datos en clientes activos e inactivos, lo que es crucial para el análisis de churn.
4. Desarrollaste funciones reutilizables para la visualización de datos, mejorando la claridad y eficiencia del análisis.
5. Utilizaste gráficos de barras y histogramas para explorar distribuciones de manera efectiva.
6. Creaste una matriz de correlación para identificar relaciones significativas entre variables.
7. Dividiste los datos en conjuntos de entrenamiento y prueba de manera adecuada.
8. Implementaste modelos de clasificación con métricas de evaluación claras.
9. Aplicaste técnicas de escalado antes del modelado, asegurando la comparabilidad de las características.
10. Exploraste el clustering para segmentar los clientes, lo cual es un enfoque avanzado y valioso.

¡Felicidades!
</div>
